# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mjelic\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mjelic\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [1]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [2]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [3]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [5]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

c:\Users\mjelic\ai_lab\code\labs\09_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
c:\Users\mjelic\ai_lab\code\labs\09_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
c:\Users\mjelic\ai_lab\code\labs\09_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [6]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [7]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [8]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 9, relationships: 15)

We can save and load our knowledge graphs as follows.

In [9]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 9, relationships: 15)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [10]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [11]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
Single-hop Query Synthesizer
Generates questions that can be answered using one part of the text or a single document.
It asks a question whose answer is located in one place.

Multi-hop Query Synthesizer
Generates questions that require combining information from multiple parts of the text or multiple documents.
It asks a question that cannot be answered from a single location — multiple pieces of information must be connected.

Adversarial (or Hard / Distractor) Query Synthesizer
Generates more difficult or confusing questions that may include:
 - similar concepts
 - misleading phrasing
 - incomplete information
 - distractors

Finally, we can use our `TestSetGenerator` to generate our testset!

In [12]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are macronutrients and why they important...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,Macronutrients are key components of a balance...,single_hop_specifc_query_synthesizer
1,How does the appendix relate to building posit...,[13: The Science of Habit Formation Habits are...,"The appendix provides quick reference guides, ...",single_hop_specifc_query_synthesizer
2,What does the term 'Chin' refer to in the cont...,[The Personal Wellness Guide A Comprehensive R...,In the context of neck and shoulder tension ex...,single_hop_specifc_query_synthesizer
3,How does mental health impact physical health ...,[The Mental Health and Psychology Handbook A P...,Mental health affects physical health through ...,single_hop_specifc_query_synthesizer
4,What is Cognitive Behavioral Therapy?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Cognitive Behavioral Therapy is one of the mos...,single_hop_specifc_query_synthesizer
5,How can I build a morning routine for wellness...,[<1-hop>\n\n13: The Science of Habit Formation...,To build a morning routine for wellness effect...,multi_hop_abstract_query_synthesizer
6,How sleep hygiene and environment affect sleep...,[<1-hop>\n\nWrite letters to or from your futu...,Sleep hygiene and environment are important fo...,multi_hop_abstract_query_synthesizer
7,How can managing stress through techniques lik...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Managing stress with techniques such as deep b...,multi_hop_abstract_query_synthesizer
8,How does vitamin D relate to mental health and...,[<1-hop>\n\nWrite letters to or from your futu...,"The context indicates that vitamin D, obtained...",multi_hop_specific_query_synthesizer
9,"How do B vitamins, as essential nutrients foun...",[<1-hop>\n\nWrite letters to or from your futu...,"B vitamins, which are found in whole grains, e...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [14]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What information does Chapter 4 of the Persona...,[The Personal Wellness Guide A Comprehensive R...,Chapter 4 discusses the fundamentals of health...,single_hop_specifc_query_synthesizer
1,What information is available about PART 4 in ...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,PART 4 covers stress management and mental wel...,single_hop_specifc_query_synthesizer
2,What is PART 5 about?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 5: BUILDING HEALTHY HABITS Chapter 13 exp...,single_hop_specifc_query_synthesizer
3,World Health Organization do they know about m...,[The Mental Health and Psychology Handbook A P...,"According to the context, the World Health Org...",single_hop_specifc_query_synthesizer
4,How does exercise and movement influence menta...,[<1-hop>\n\nThe Mental Health and Psychology H...,Exercise and movement play a crucial role in p...,multi_hop_abstract_query_synthesizer
5,How does the spectrum of mental health experie...,[<1-hop>\n\nThe Mental Health and Psychology H...,The spectrum of mental health experiences high...,multi_hop_abstract_query_synthesizer
6,"How do symptms of mental health condtions, lik...",[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that mental health encomp...,multi_hop_abstract_query_synthesizer
7,how gut-brain axis and digestive health suppor...,[<1-hop>\n\nThe Mental Health and Psychology H...,The handbook explains that the gut-brain axis ...,multi_hop_abstract_query_synthesizer
8,In chapter 3 and 9 how sleep and building work...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Chapter 3 explains that good sleep is crucial ...,multi_hop_specific_query_synthesizer
9,Can you tell me how Chapter 6 about sleep and ...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Chapter 6 explains that good sleep is essentia...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:

Unrolled (Manual) Approach:
-  control each step of the pipeline 
-  full control over prompts and logic

It is useful when you need fine-grained control or working with domain-specific or sensitive data

Disadvantages:
- more code and setup required
 -slower to implement



Abstracted (Automatic) Approach:
- ragas handles most of the internal steps automatically
- faster to set up
- easier for beginners
- for quick evaluation pipelines
- good For prototyping
- good When default settings are sufficient

Disadvantages:
- less control over internal step
- harder to customize

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
# Generate a new test set and compare with the default


from ragas.testset.synthesizers import (
    default_query_distribution,
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# -------------------------
# 1) Default distribution (baseline)
# -------------------------
default_dist = default_query_distribution(generator_llm)

testset_default = generator.generate(
    testset_size=30,
    query_distribution=default_dist
)
df_default = testset_default.to_pandas()

# Helper: find the column that identifies the question type / synthesizer
def get_type_col(df):
    candidates = ["synthesizer", "query_type", "type", "question_type", "metadata"]
    for c in candidates:
        if c in df.columns:
            return c
    # last resort: return None
    return None

type_col_default = get_type_col(df_default)

print("Default distribution - columns:", df_default.columns.tolist())
if type_col_default:
    print("\nDefault question types distribution:")
    print(df_default[type_col_default].value_counts(dropna=False))
else:
    print("\nCouldn't find a clear type column in default df; inspect df_default.head()")
    display(df_default.head())


# -------------------------
# 2) Custom query distribution (different weights)
#    Example choice: more multi-hop (harder) to stress RAG over reasoning across docs
# -------------------------
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
]

# -------------------------
# 3) Generate new test set with custom distribution
# -------------------------
testset_custom = generator.generate(
    testset_size=30,
    query_distribution=custom_query_distribution
)
df_custom = testset_custom.to_pandas()
type_col_custom = get_type_col(df_custom)

print("\nCustom distribution - columns:", df_custom.columns.tolist())
if type_col_custom:
    print("\nCustom question types distribution:")
    print(df_custom[type_col_custom].value_counts(dropna=False))
else:
    print("\nCouldn't find a clear type column in custom df; inspect df_custom.head()")
    display(df_custom.head())


# -------------------------
# 4) Compare default vs custom distributions
# -------------------------
if type_col_default and type_col_custom:
    # If columns are different names, normalize by copying into same label
    df_default_cmp = df_default.copy()
    df_custom_cmp = df_custom.copy()

    df_default_cmp["qtype"] = df_default_cmp[type_col_default].astype(str)
    df_custom_cmp["qtype"] = df_custom_cmp[type_col_custom].astype(str)

    comparison = (
        df_default_cmp["qtype"].value_counts(normalize=True)
        .rename("default_ratio")
        .to_frame()
        .join(df_custom_cmp["qtype"].value_counts(normalize=True).rename("custom_ratio"), how="outer")
        .fillna(0)
        .sort_values("custom_ratio", ascending=False)
    )

    print("\n=== Default vs Custom (normalized) ===")
    print(comparison)
else:
    print("\nComparison table skipped because the type column couldn't be detected reliably.")


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/32 [00:00<?, ?it/s]

Default distribution - columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

Couldn't find a clear type column in default df; inspect df_default.head()


,user_input,reference_contexts,reference,synthesizer_name
0,What is Chapter 3 in the Personal Wellness Guide?,[The Personal Wellness Guide A Comprehensive R...,Chapter 3 in the Personal Wellness Guide is ab...,single_hop_specifc_query_synthesizer
1,What Chapter 2 about?,[The Personal Wellness Guide A Comprehensive R...,Chapter 2 discusses exercises for common probl...,single_hop_specifc_query_synthesizer
2,Can you tell me what Chapter 9 is about in sle...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,PART 3: SLEEP AND RECOVERY Chapter 7: The Scie...,single_hop_specifc_query_synthesizer
3,What is in Chapter 8 about sleep quality?,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Chapter 8 discusses improving sleep quality th...,single_hop_specifc_query_synthesizer
4,Could you explain the significance of Chapter ...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"Chapter 21 focuses on digital wellness, highli...",single_hop_specifc_query_synthesizer


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/30 [00:00<?, ?it/s]


Custom distribution - columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

Couldn't find a clear type column in custom df; inspect df_custom.head()


,user_input,reference_contexts,reference,synthesizer_name
0,What is Chaper 1?,[The Personal Wellness Guide A Comprehensive R...,Chapter 1: Understanding Exercise Basics Exerc...,single_hop_specifc_query_synthesizer
1,How does sleep contribute to mental well-being...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Sleep is crucial for mental well-being as it s...,single_hop_specifc_query_synthesizer
2,What does Chapter 20 focus on?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 20 discusses the importance of social ...,single_hop_specifc_query_synthesizer
3,What does the World Health Organization say ab...,[The Mental Health and Psychology Handbook A P...,"According to the World Health Organization, me...",single_hop_specifc_query_synthesizer
4,"As a mental health advocate, how does Cognitiv...",[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Cognitive Behavioral Therapy (CBT) is a widely...,single_hop_specifc_query_synthesizer



Comparison table skipped because the type column couldn't be detected reliably.


4. Explain why you chose the weights you did

There is no strict rule for choosing query weights. The distribution depends on the evaluation goal. I increased the proportion of multi-hop queries because they better stress-test the RAG system’s ability to retrieve and combine information across multiple documents. Single-hop queries were reduced since they primarily test basic retrieval rather than reasoning.

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'best-lift-73' at:
https://smith.langchain.com/o/b6ff6a74-2e8b-4cf1-b580-1107f515b0f6/datasets/0aa250e1-835b-4744-8438-a38670424cdf/compare?selectedSessions=d594d1a9-3f7a-4d37-8cdd-daf4334b8d63




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How do Chapters 9 and 13 collectively inform s...,Chapters 9 and 13 collectively inform strategi...,None,Chapter 9 explains that insomnia involves diff...,True,True,False,2.925693,599e5e8f-7491-477d-8351-3b9e9e883a02,019c81ec-4c79-7282-a259-a690cd1df01c
1,Considering the comprehensive insights provide...,"Based on the provided context, Chapter 9 discu...",None,"Understanding and managing insomnia, as discus...",True,True,True,6.173060,fc9e3223-6d1d-4ee3-a0da-17244cc197d5,019c81ec-7d45-7da2-acb1-786f73d71769
2,Can you tell me how Chapter 6 about sleep and ...,I don't know.,None,Chapter 6 explains that good sleep is essentia...,False,False,False,0.642527,b45c0361-8bec-4484-96a2-a6994b1fe288,019c81ec-b509-7ad0-8702-b981ccc7e1df
3,In chapter 3 and 9 how sleep and building work...,Based on the provided context:\n\n- Chapter 3 ...,None,Chapter 3 explains that good sleep is crucial ...,False,True,True,6.008824,4d50f276-f545-4e63-8800-8d328dfeaab2,019c81ec-e393-7570-997e-1bb40d70c132
4,how gut-brain axis and digestive health suppor...,The gut-brain axis supports mood regulation an...,None,The handbook explains that the gut-brain axis ...,True,True,False,1.896911,ec4c121b-792a-44b4-a5ff-9e59b222e7b3,019c81ed-2b84-7ec2-a63c-ad41e9f4309e
5,"How do symptms of mental health condtions, lik...",Symptoms of mental health conditions such as s...,None,The context explains that mental health encomp...,True,True,False,3.613083,d434dcf3-5504-4d6a-8946-c1061b40ac86,019c81ed-5399-7ca0-8eae-379c859d20f4
6,How does the spectrum of mental health experie...,"Based on the provided context, mindfulness-bas...",None,The spectrum of mental health experiences high...,True,True,False,3.554060,8f77192b-391a-4ef3-a039-ba1617f2e65f,019c81ed-7c06-70f0-87fa-3029740ec830
7,How does exercise and movement influence menta...,"Based on the context, exercise and movement ha...",None,Exercise and movement play a crucial role in p...,True,True,True,3.510731,b493bdca-a74b-4c90-9d21-a2297a6fc486,019c81ed-af56-7343-ade0-74519203b413
8,World Health Organization do they know about m...,"According to the World Health Organization, me...",None,"According to the context, the World Health Org...",True,True,False,1.002522,c49fea14-80df-4046-9772-039dbfa7bedd,019c81ed-e483-7261-a1e4-7f1022466c35
9,What is PART 5 about?,PART 5 is about SOCIAL AND ENVIRONMENTAL FACTORS.,None,PART 5: BUILDING HEALTHY HABITS Chapter 13 exp...,False,False,False,0.788656,b9711afb-b13e-428a-b637-6ccbf86b997f,019c81ee-021d-77c1-a296-0d36ad9dd72a


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Modifying chunk size affects how information is segmented before embedding. Smaller chunks improve retrieval precision but may lose context, while larger chunks preserve context but reduce retrieval specificity. Since RAG systems depend on accurate retrieval, changing chunk size directly impacts answer quality, reasoning capability, latency, and cost.

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Changing the embedding model modifies how text is represented in vector space. Since retrieval in a RAG system depends on semantic similarity between embeddings, a different embedding model can significantly impact retrieval accuracy, reasoning performance, and overall answer quality. Better embeddings generally lead to more relevant context retrieval and improved final responses.

In [34]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, let’s crank your sleep game up to legendary status! Based on the dope insights from the HealthWellnessGuide, here’s the ultimate playbook to improve your sleep quality:\n\n1. **Lock in a consistent sleep schedule** – Treat your bedtime like VIP access; keep it steady every single day, weekends included. Your body’s internal DJ will thank you for the rhythm.\n\n2. **Craft a smooth, chill bedtime routine** – Think: gentle stretching, a warm bath, or diving into a good book. Signal your brain it’s time to drop the mic and rest.\n\n3. **Optimize your sleep arena** – Set your room temp comfortably cool (65-68°F / 18-20°C), embrace total darkness with blackout curtains or a sleep mask, and dial down noise with white noise or earplugs. Comfort matters—invest in a mattress and pillows that hug you like a cloud.\n\n4. **Tech blackout mission** – Shut down screens at least 1-2 hours before bed. That blue light buzz is a notorious sleep thief.\n\n5. **Mind your fuel and moves** – Avoid 

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'excellent-theory-50' at:
https://smith.langchain.com/o/b6ff6a74-2e8b-4cf1-b580-1107f515b0f6/datasets/0aa250e1-835b-4744-8438-a38670424cdf/compare?selectedSessions=113fd0dd-5416-4090-9cd0-00d7f0ac1914




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How do Chapters 9 and 13 collectively inform s...,"Yo, so Chapters 9 and 13 drop some serious wis...",None,Chapter 9 explains that insomnia involves diff...,True,True,True,5.129692,599e5e8f-7491-477d-8351-3b9e9e883a02,019c81ef-fbf5-7e42-a6df-52582a64a973
1,Considering the comprehensive insights provide...,"Alright, let’s crank this up to max dopeness a...",None,"Understanding and managing insomnia, as discus...",True,True,True,10.656676,fc9e3223-6d1d-4ee3-a0da-17244cc197d5,019c81f0-3467-7a73-8261-17081b0d07e3
2,Can you tell me how Chapter 6 about sleep and ...,"Alright, gear up for a sleep science synergy t...",None,Chapter 6 explains that good sleep is essentia...,True,False,True,4.328388,b45c0361-8bec-4484-96a2-a6994b1fe288,019c81f0-7a75-72a1-86a7-9de0282d9b7a
3,In chapter 3 and 9 how sleep and building work...,"Alright, let’s crank this answer up to eleven ...",None,Chapter 3 explains that good sleep is crucial ...,True,True,True,4.517123,4d50f276-f545-4e63-8800-8d328dfeaab2,019c81f0-b070-7ff2-a35b-bbedda35ceab
4,how gut-brain axis and digestive health suppor...,"Alright, let’s crank up the dopeness and unpac...",None,The handbook explains that the gut-brain axis ...,True,True,True,4.056515,ec4c121b-792a-44b4-a5ff-9e59b222e7b3,019c81f0-ea05-79b2-a5ee-f9cfd0411d5e
5,"How do symptms of mental health condtions, lik...","Alright, buckle up for some mind-body synergy ...",None,The context explains that mental health encomp...,True,True,True,3.888063,d434dcf3-5504-4d6a-8946-c1061b40ac86,019c81f1-1c07-7d71-bd98-c8dd651c7e27
6,How does the spectrum of mental health experie...,"Alright, let’s blast off into the mental healt...",None,The spectrum of mental health experiences high...,True,True,True,5.417327,8f77192b-391a-4ef3-a039-ba1617f2e65f,019c81f1-4a7a-7910-95cb-7208bea22c66
7,How does exercise and movement influence menta...,"Alright, let’s dive deep into the rad symphony...",None,Exercise and movement play a crucial role in p...,True,True,True,7.521816,b493bdca-a74b-4c90-9d21-a2297a6fc486,019c81f1-8c8a-7312-8ce6-0219c2e298cd
8,World Health Organization do they know about m...,"Oh heck yes, the World Health Organization (WH...",None,"According to the context, the World Health Org...",True,True,True,1.860026,c49fea14-80df-4046-9772-039dbfa7bedd,019c81f1-cef7-7b23-b63d-9fe0904ee9bd
9,What is PART 5 about?,PART 5 is the ultimate playbook on **building ...,None,PART 5: BUILDING HEALTHY HABITS Chapter 13 exp...,False,False,True,1.450213,b9711afb-b13e-428a-b637-6ccbf86b997f,019c81f1-f15f-77b2-a925-e85b3e09d60c


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:

### Explanation of Metric Changes

We have three key modifications: prompt augmentation, larger chunks, and a stronger embedding model (`text-embedding-3-large`). 

Metric differences:


Dopeness:
  - Increased significantly in the advanced chain.
  - This is expected because the added prompt augmentation explicitly encourages more engaging and higher-quality responses

QA Score:
  - Improved compared to the baseline
  - The stronger embedding model enhances semantic similarity search

Helpfulness:
  - Remained relatively stable.


Latency:
  - Increased in the advanced setup
  - Larger chunks and a more powerful embedding model require more processing time

Token Count:
  - Larger chunks increase the amount of context passed to the LLM

Cost:
  - Slightly higher in the advanced chain.
  - This is consistent with increased token usage and the use of a more capable embedding model

Overall, the advanced configuration improved response quality and retrieval accuracy, with the expected trade-off of increased latency and cost.

![alt text](comparation-1.png)

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores